# Hyperparameter optimization with Optuna

This section performs hyperparameter optimization using Optuna. The search evaluates different ResNet depths, learning rates, weight decay values, dropout rates, learning rate scheduler settings, and optimizer choices.

Each trial trains a model on the training dataset and evaluates performance on the validation dataset. Optuna pruning is used to stop underperforming trials early. The objective function returns negative validation accuracy so that minimizing the objective corresponds to maximizing validation accuracy.

## 1. Import required libraries

This section imports the core Python packages used throughout the notebook. PyTorch is used for dataset handling, model construction, optimization, and inference. Torchvision provides the ResNet backbone. Scikit-learn is used for performance evaluation, while Matplotlib and Seaborn are used to generate summary figures.

In [ ]:
import os
from pathlib import Path
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

## 2. Define the reduced tensor dataset

The input data for this model are stored as PyTorch `.pt` tensor files. Each file corresponds to one reduced image sequence and is assigned to a class based on its parent folder.

The logarithmic classification scheme uses five ordered classes:

- `0.undetectable`
- `1.low`
- `2.medium`
- `3.high`
- `4.very high`

Each tensor is expected to contain 7 temporal channels. These channels represent a reduced version of the original image sequence and preserve selected spatial-temporal information relevant to viral load classification.

The custom `PTDataset` class scans the class folders, associates each file with its class label, loads the tensors, removes an extra singleton channel dimension when present, and returns `(tensor, label)` pairs for use with PyTorch dataloaders.

In [ ]:
class PTDataset(Dataset):
    def __init__(self, root_dir, target_size=(500, 500), transform=None):
        """
        Args:
            root_dir (str): Path to the reduced dataset directory.
            target_size (tuple): Kept for compatibility, but not used because
                                 reduced tensors are already resized.
            transform (callable, optional): Optional transformations.
        """
        self.root_dir = root_dir
        self.target_size = target_size
        self.transform = transform
        self.classes = ['0.undetectable', '1.low', '2.medium', '3.high', '4.very high']

        # Collect all file paths and labels
        self.file_list = []
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):
                if file.endswith('.pt'):
                    full_path = os.path.join(class_path, file)
                    class_index = self.classes.index(label)
                    self.file_list.append((full_path, class_index))

        # Pre-load everything into memory
        self.data_list = []
        for file_path, label in self.file_list:
            # Load reduced tensor from disk
            tensor_data = torch.load(file_path, map_location='cpu')

            # Reduced files should already be [1, 7, H, W] or [7, H, W]
            if tensor_data.dim() == 4 and tensor_data.shape[0] == 1:
                tensor_data = tensor_data.squeeze(0)  # [7, H, W]

            # Optional transform
            if self.transform:
                tensor_data = self.transform(tensor_data)

            # Store (tensor, label)
            self.data_list.append((tensor_data, label))

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

## 3. Configure dataset paths

The following section specifies the locations of the training, validation, and testing datasets. Each dataset folder is expected to contain one subfolder per class.

```python
/workspace/data/logarithmic/Training
/workspace/data/logarithmic/Validation
/workspace/data/logarithmic/Testing

In [ ]:
DATA_ROOT = Path("/workspace/data/logarithmic")
OUTPUT_ROOT = Path("/workspace/outputs")
MODEL_PATH = OUTPUT_ROOT / "logarithmic_resnet18.pth"

TRAIN_DIR = DATA_ROOT / "Training"
VAL_DIR = DATA_ROOT / "Validation"
TEST_DIR = DATA_ROOT / "Testing"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_dataset = PTDataset(root_dir=TRAIN_DIR, target_size=(500, 500))
val_dataset = PTDataset(root_dir=VAL_DIR, target_size=(500, 500))
test_dataset = PTDataset(root_dir=TEST_DIR, target_size=(500, 500))

## 4. Hyperparameter optimization with Optuna

This section performs hyperparameter optimization using Optuna. The search evaluates different ResNet depths, learning rates, weight decay values, dropout rates, learning rate scheduler settings, and optimizer choices.

Each trial trains a model on the training dataset and evaluates performance on the validation dataset. Optuna pruning is used to stop underperforming trials early. The objective function returns negative validation accuracy so that minimizing the objective corresponds to maximizing validation accuracy.

In [ ]:


def get_resnet_model(model_depth=34, dropout=0.5, num_classes=5, input_channels=7):
    """Returns a ResNet model with specified depth and dropout."""
    if model_depth == 18:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    elif model_depth == 34:
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    elif model_depth == 50:
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    else:
        raise ValueError("Invalid ResNet depth. Choose from 18, 34, or 50.")
    
    model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)  # Modify for 7 channels
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model

def evaluate_model(model, loader, criterion, device):
    """Evaluates the model on a dataset."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc

def objective(trial):
    """Optuna optimization function with pruning."""
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'dropout': trial.suggest_float('dropout', 0.0, 0.9),
        'gamma_rate': trial.suggest_float('gamma_rate', 0.9, 0.99),
        'optimizer': trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'ASGD', 'LBFGS'])
    }
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = get_resnet_model(params['model_depth'], params['dropout']).to(device)
    criterion = nn.CrossEntropyLoss()
    
    if params['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    elif params['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'], momentum=0.9)
    elif params['optimizer'] == 'ASGD':
        optimizer = optim.ASGD(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    elif params['optimizer'] == 'LBFGS':
        optimizer = optim.LBFGS(model.parameters(), lr=params['learning_rate'])  # Removed weight_decay
    else:
        raise ValueError("Unsupported optimizer type")
    
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=params['gamma_rate']) if params['gamma'] else None
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Training loop with pruning
    for epoch in range(25):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            def closure():
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                return loss
            
            if params['optimizer'] == 'LBFGS':
                optimizer.step(closure)
            else:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
        
        if scheduler:
            scheduler.step()

        # Evaluate on validation set
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        
        # Report intermediate result for pruning
        trial.report(val_acc, epoch)

        # Stop early if performance is worse than previous trials at this step
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -val_acc  # Minimizing negative accuracy to maximize positive accuracy

# Run Optuna study with Median Pruner
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
study.optimize(objective, n_trials=200)

# Display results
best_trial = study.best_trial
print("\nBest trial:")
print(f"  Value: {best_trial.value}")  # Converting back to positive accuracy
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")
